# 💵 **2024 연말정산 안내문 기반 RAG 질의응답 시스템 (LangChain)**

✅ 프로젝트 설명
- 목표: 국세청 ‘2024 연말정산 신고 안내’ PDF를 근거로, 질문을 입력하면 문서 근거를 찾아서 LLM이 답하도록 만들기

- 핵심 아이디어:
    - PDF 문서 훑어보고 전체적인 형식을 파악
    - 문서를 잘게 나누기(청킹)
    - 임베딩해서 벡터DB에 저장
    - 질문이 들어오면 관련 문서를 찾기
    - LLM이 그 문서를 참고해서 답변하게 만들기
<br>

✅ 문서 특징
- 표가 많다
- 일반 PDF 텍스트 파서로 읽으면 표가 뭉개질 수 있다
- 그래서 HTML 구조를 살리는 방식이 중요하다
<br>

✅ 내가 선택한 전체흐름

- 문서 로드: Upstage Document Parse (HTML)
    - 이유: 세무 문서는 표가 많아서, 텍스트 로더가 표를 “한 줄 텍스트 덩어리”로 망가뜨리면 RAG 정확도가 급락함

- 정제(노이즈 제거)
    - 이유: 헤더/푸터 반복 문구는 검색을 “교란”함 (질문이 아니라 헤더가 자꾸 걸림)

- 청킹: Semantic Chunking (BGE-m3 기반)
    - 이유: 512/1000자처럼 기계적으로 자르면 표 중간이 찢어져서 맥락이 죽음

- 최적화: Smart Diet (긴 HTML만 Markdown 변환)
    - 이유: HTML 태그는 토큰을 많이 잡아먹음 → 긴 청크만 가볍게 만들어서 속도/정확도 개선 기대

- 검색: Hybrid (Dense + BM25)
    - 이유: 세무는 “용어 정확도(월세/주택임차차입금)”가 중요해서 키워드 검색이 같이 있어야 강해짐

- 재순위: BGE Reranker
    - 이유: 처음 검색 Top-K는 후보군. 그중 진짜 정답을 다시 골라주는 단계가 있으면 정확도가 크게 오름

- 생성: EXAONE 3.5 7.8B (4bit)
    - 이유: 한국어 세무 용어에 강한 계열 + Colab GPU에서 돌리기 위해 양자화

◻️ 구글드라이브 & 경로체크

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

PDF_PATH = "/content/drive/MyDrive/정연경 강사/2024년+원천징수의무자를+위한+연말정산+신고안내.pdf"

# WHY: 뒤에서 Upstage 로더든 PyMuPDF든 "경로가 틀리면 전부 실패"하니까, 초반에 확실히 체크해두는 게 디버깅 시간을 가장 줄임.
assert os.path.exists(PDF_PATH), f"PDF 파일이 없습니다: {PDF_PATH}"

print("PDF OK:", os.path.basename(PDF_PATH))
print("size(MB):", os.path.getsize(PDF_PATH) / (1024*1024))


◻️ 라이브러리 임포트

In [ ]:
import os, time, json, pickle, re, random, warnings
from collections import Counter
from typing import List, Dict, Any

import numpy as np
import pandas as pd

import torch
import matplotlib.pyplot as plt

from getpass import getpass
from dotenv import load_dotenv

from bs4 import BeautifulSoup
import html2text

from pypdf import PdfReader, PdfWriter

# LangChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

from langchain_community.retrievers import BM25Retriever
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

# Upstage
from langchain_upstage import UpstageDocumentParseLoader, UpstageEmbeddings

# Embedding / Reranker
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from sentence_transformers import CrossEncoder

# Vector DB
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

# LLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# 형태소
from kiwipiepy import Kiwi

warnings.filterwarnings("ignore")

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


◻️ 폰트 설치

In [ ]:
# 폰트 캐시 제거 (필수)
!rm -rf ~/.cache/matplotlib

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings(action='ignore')

# 폰트 파일 경로 (Google Drive/MyDrive/fonts/BMHANNAPro.ttf)
path = '/content/drive/MyDrive/fonts/BMHANNAPro.ttf'

# 폰트 이름 지정
# 파일 이름은 BMHANNAPro이지만, Matplotlib에서 인식할 이름은 보통 'BM HANNA Pro'입니다.
font_name = 'BM HANNA Pro'

# Matplotlib에 폰트 등록
fm.fontManager.addfont(path)

# Matplotlib 기본 폰트 설정
plt.rc('font', family=font_name)

print(f"✅ Matplotlib 폰트가 '{font_name}'로 설정되었습니다.")

◻️ API KEY 세팅

In [ ]:
load_dotenv()

# WHY: 키를 코드에 하드코딩하면 공유/제출 시 보안사고가 나고 깃허브에도 업로드 자체가 안되었음. getpass는 입력값을 화면에 안 찍어줘서 가장 안전한 편.
os.environ["UPSTAGE_API_KEY"]   = getpass("Upstage API Key: ")
os.environ["LANGCHAIN_API_KEY"] = getpass("LangSmith API Key (optional): ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "미션14-연말정산-RAG"
os.environ["HF_TOKEN"]          = getpass("HuggingFace Token: ")

print("keys loaded (masked)")
for k in ["UPSTAGE_API_KEY","LANGCHAIN_API_KEY","HF_TOKEN"]:
    v = os.environ.get(k, "")
    print(k, "OK" if v else "MISSING")


◻️ 왜 Upstage를 사용하는지 1페이지를 로더해보고 비교

In [ ]:
# WHY: 전체 문서를 바로 돌리면 비용/시간이 큼. 그래서 딱 1페이지만 뽑아서 "표/제목 구조가 살아있는지" 먼저 검증하는 게 합리적.

TARGET_PAGE_INDEX = 13  # 0-based

# 1) PyMuPDF 방식
loader_basic = PyMuPDFLoader(PDF_PATH)
docs_basic = loader_basic.load()
basic_text = docs_basic[TARGET_PAGE_INDEX].page_content

# 2) Upstage 방식: 1페이지만 임시 PDF로 만들어서 API 호출 최소화
temp_pdf = "temp_one_page.pdf"
reader = PdfReader(PDF_PATH)
writer = PdfWriter()
writer.add_page(reader.pages[TARGET_PAGE_INDEX])
with open(temp_pdf, "wb") as f:
    writer.write(f)

loader_up = UpstageDocumentParseLoader(
    temp_pdf,
    split="none",             # WHY: 1페이지만 가져오니까 더 쪼갤 필요 없음
    output_format="html"      # WHY: 표 구조(rowspan/colspan)를 보존하려면 html이 유리
)
up_docs = loader_up.load()
up_html = up_docs[0].page_content

# 태그 제거(비교용)
soup = BeautifulSoup(up_html, "html.parser")
up_text = soup.get_text(separator="\n", strip=True)

os.remove(temp_pdf)

print("---- PyMuPDF (앞 300자) ----")
print(basic_text[:300], "...\n")

print("---- Upstage (태그 제거, 앞 300자) ----")
print(up_text[:300], "...\n")

print("length stats")
print("PyMuPDF chars:", len(basic_text))
print("Upstage text chars:", len(up_text))
print("Upstage html chars:", len(up_html))

# WHY: 육안으로 표/목차/제목이 "살아있는 느낌"인지 확인하는 게 중요. RAG는 '문서 품질'이 절반 이상이라, 로더 선택이 그냥 성능을 결정한다.


◻️ 전제 문서 로드 - Upstage & 페이지 단위

In [ ]:
start = time.time()

loader = UpstageDocumentParseLoader(
    PDF_PATH,
    split="page",           # WHY: EDA/정제에서 페이지 단위가 가장 다루기 쉬움
    output_format="html"    # WHY: 표 구조 보존
)

docs = loader.load()

print("pages:", len(docs))
print("time(sec):", round(time.time() - start, 2))

# WHY: 제출/재현성 때문에 "중간 결과 저장"을 습관처럼 해두면 좋음. API 재호출(비용/시간)을 줄이는 효과도 큼.
pickle_path = "/content/drive/MyDrive/2024_tax_guide_docs.pkl"
json_path   = "/content/drive/MyDrive/2024_tax_guide_docs.json"

with open(pickle_path, "wb") as f:
    pickle.dump(docs, f)

serializable = [{"page": i+1, "content": d.page_content, "metadata": d.metadata} for i, d in enumerate(docs)]
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

print("saved:", pickle_path)
print("saved:", json_path)


◻️ EDA
- HTML vs Text 길이(정보밀도)로 문서가 표 중심인치 체크

In [ ]:
html_lengths, text_lengths, densities = [], [], []

for d in docs:
    html = d.page_content
    h_len = len(html)

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    t_len = len(text)

    density = (t_len / h_len) if h_len > 0 else 0

    html_lengths.append(h_len)
    text_lengths.append(t_len)
    densities.append(density)

df_stats = pd.DataFrame({
    "page_no": [d.metadata.get("page", i+1) for i, d in enumerate(docs)],
    "html_length": html_lengths,
    "text_length": text_lengths,
    "density": densities,
})

print("pages:", len(df_stats))
print("avg html:", int(df_stats["html_length"].mean()))
print("avg text:", int(df_stats["text_length"].mean()))
print("avg density:", round(df_stats["density"].mean(), 3))

# WHY: density가 너무 낮은 페이지는 "태그는 많은데 실제 텍스트는 빈약"한 경우가 많고, 이런 페이지는 청킹/검색/생성에서 효율이 떨어질 수 있어 별도 처리 후보가 됨.

plt.figure(figsize=(10,4))
plt.hist(df_stats["html_length"], bins=50, alpha=0.5, label="html_length")
plt.hist(df_stats["text_length"], bins=50, alpha=0.5, label="text_length")
plt.legend()
plt.title("HTML vs Text length distribution")
plt.show()

empty_pages = df_stats[df_stats["text_length"] < 50]
print("text<50 pages:", len(empty_pages))
if len(empty_pages) > 0:
    print(empty_pages[["page_no", "text_length"]].head(20))


◻️ 헤더 / 푸터 패턴 찾기
- 반복 노이즈 후보

In [ ]:
top_lines, bottom_lines = [], []
clean_candidates = []

for d in docs:
    soup = BeautifulSoup(d.page_content, "html.parser")
    text = soup.get_text(separator="\n", strip=True)

    # WHY: 너무 짧은 페이지는 보통 쪽번호/빈칸/서식만 있는 페이지라 검색에 도움보다 방해가 될 확률이 큼.
    if len(text) < 50:
        continue

    clean_candidates.append(d)

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if len(lines) >= 3:
        top_lines.extend(lines[:3])
        bottom_lines.extend(lines[-3:])
    else:
        top_lines.extend(lines)
        bottom_lines.extend(lines)

top_counter = Counter(top_lines).most_common(10)
bottom_counter = Counter(bottom_lines).most_common(10)

print("header suspects")
for phrase, cnt in top_counter:
    print(cnt, phrase)

print("\nfooter suspects")
for phrase, cnt in bottom_counter:
    print(cnt, phrase)

# WHY: RAG 검색은 "질문과 관련"을 찾는 건데, 헤더가 수백 번 반복되면 '질문과 무관한데도' 항상 상위에 뜨는 교란이 됨. ➔ 결국 헤더 제거는 검색 품질을 직접 올리는 작업.


◻️ 데이터 정제
- 빈 페이지 제거 & 헤더 문자열 제거

In [ ]:
GARBAGE_PATTERNS = [
    "원천징수의무자를 위한",
    "2024년 연말정산 신고안내",
    "210mm×297mm",
    "[백상지 80g/㎡(재활용품)]",
]

filtered_docs = []
removed_short = 0
replaced_count = 0

for d in docs:
    # 1) 빈 페이지 제거 기준(텍스트 기준)
    soup = BeautifulSoup(d.page_content, "html.parser")
    txt = soup.get_text(separator="", strip=True)
    if len(txt) < 50:
        removed_short += 1
        continue

    # 2) 문자열 치환
    # WHY: HTML 구조 자체는 최대한 건드리지 않고, "반복 텍스트"만 제거해서 검색 교란을 줄이려는 목적.
    content = d.page_content
    for pat in GARBAGE_PATTERNS:
        if pat in content:
            content = content.replace(pat, "")
            replaced_count += 1

    # 3) 원본을 직접 수정하기보다 새 Document로 만드는 게 안전함
    # WHY: 어떤 라이브러리는 doc 객체를 공유/참조할 수 있어서, inplace 수정이 나중에 디버깅을 어렵게 만들 수 있음.
    filtered_docs.append(Document(page_content=content, metadata=dict(d.metadata)))

print("original pages:", len(docs))
print("removed short pages:", removed_short)
print("replaced patterns:", replaced_count)
print("final pages:", len(filtered_docs))

docs = filtered_docs  # 정제본 사용


◻️ 토큰 분포로 청킹 크기 체크

In [ ]:
TARGET_MODELS = [
    "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct",
    "upstage/solar-10.7b-instruct-v1.0",
    "beomi/Llama-3-Open-Ko-8B",
]

tok = None
loaded_name = None

for model_id in TARGET_MODELS:
    try:
        tok = AutoTokenizer.from_pretrained(model_id, token=os.environ.get("HF_TOKEN"), trust_remote_code=True)
        loaded_name = model_id
        break
    except Exception as e:
        print("tokenizer failed:", model_id, "|", str(e)[:120])

assert tok is not None, "토크나이저 로드 실패"

token_counts = []
for d in docs:
    # WHY: 실제 LLM 컨텍스트 제한은 '글자수'가 아니라 '토큰수'로 결정됨. 그래서 청킹 기준을 토큰 감각으로 잡는 게 실전에서 훨씬 안정적이라고 함.
    token_counts.append(len(tok.encode(d.page_content, add_special_tokens=False)))

token_counts = np.array(token_counts)
print("pages:", len(token_counts))
print("min:", token_counts.min(), "max:", token_counts.max(), "mean:", token_counts.mean(), "median:", np.median(token_counts))

plt.figure(figsize=(10,4))
plt.hist(token_counts, bins=50, alpha=0.8)
plt.title(f"Token distribution ({loaded_name})")
plt.show()


◻️ 임베딩 + Semantic Chunking

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# WHY: SemanticChunker는 "임베딩 유사도 변화"를 보고 끊음. 즉, 글자수 기준이 아니라 '의미가 바뀌는 지점'에서 끊는 방식.
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)

raw_chunks = semantic_splitter.split_documents(docs)
print("chunks:", len(raw_chunks))



◻️ Smart Diet
- 긴 HTML만 Markdown으로 가볍게 만들기

In [ ]:
def smart_diet_pipeline(chunks: List[Document], diet_threshold: int = 2000) -> List[Document]:
    """
    긴 HTML 청크는 태그가 토큰을 많이 잡아먹는다.
    그래서 '충분히 긴 것'만 골라 Markdown으로 변환해 가볍게 만든다.

    WHY: 짧은 청크까지 변환하면 오히려 변환 오차/형식 깨짐만 늘 수 있음.
    """
    h2t = html2text.HTML2Text()
    h2t.ignore_links = False
    h2t.ignore_images = True
    h2t.ignore_emphasis = False
    h2t.body_width = 0  # WHY: 자동 줄바꿈을 막아 표 레이아웃 깨짐을 줄임

    out = []
    converted = 0
    saved = 0

    for d in chunks:
        content = d.page_content
        if len(content) >= diet_threshold and ("<td" in content or "<table" in content or "<div" in content):
            try:
                md = h2t.handle(content)
                new_meta = dict(d.metadata)
                new_meta["conversion"] = "html_to_markdown"
                new_meta["original_length"] = len(content)
                new_meta["new_length"] = len(md)

                out.append(Document(page_content=md, metadata=new_meta))
                converted += 1
                saved += (len(content) - len(md))
            except Exception:
                out.append(d)
        else:
            out.append(d)

    print("converted:", converted, "| saved chars:", saved)
    return out

final_chunks = smart_diet_pipeline(raw_chunks, diet_threshold=2000)
print("final chunks:", len(final_chunks))


◻️ Vector DB(Qdrant) + BM25 + Hybrid(Ensemble)

In [ ]:
# Dense: 의미 검색(임베딩)
# Sparse: 키워드 검색(BM25)
'''
세무 문서는 전문 용어가 많아서 "키워드 정확히 맞추는 능력"이 매우 중요하다고 함.
Dense만 쓰면 비슷한 문장에 끌려가고, Sparse만 쓰면 동의어/표현차이를 놓침.
➔ 둘을 섞는 하이브리드가 실전에서 강함.
'''

qdrant_client = QdrantClient(":memory:")
collection_name = "tax_guide_2024"

# WHY: bge-m3 임베딩 차원은 1024로 알려져 있고(모델 스펙), 차원이 안 맞으면 Qdrant가 에러를 냄.
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=collection_name,
    embedding=embeddings,
)

vectorstore.add_documents(final_chunks)
print("qdrant upsert done:", len(final_chunks))

dense_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(final_chunks)
bm25_retriever.k = 5

# EnsembleRetriever import 경로가 환경마다 다를 수 있어 안전하게 처리
try:
    from langchain.retrievers import EnsembleRetriever
except Exception:
    from langchain_community.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.3, 0.7]  # WHY: 의미 0.7 + 키워드 0.3 (세무는 키워드도 중요해서 0으로 못 둠)
)

query = "월세 세액공제 한도가 얼마야?"
hits = ensemble_retriever.invoke(query)

print("retrieved:", len(hits))
for i, d in enumerate(hits[:5], 1):
    print(f"\nRank {i} | page={d.metadata.get('page')} | fmt={d.metadata.get('conversion','original')}")
    print(d.page_content[:160].replace("\n", " "), "...")


◻️ Reranker
- 후보 Top-K를 “진짜 정답 순서”로 다시 정렬

In [ ]:
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    trust_remote_code=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

def rerank_docs(query: str, retrieved_docs: List[Document], top_k: int = 3) -> List[Document]:
    """
    WHY: Hybrid 검색은 후보를 넓게 잡는 단계.
         Reranker는 (질문, 문서) 쌍을 더 강하게 비교해서 '정답 같은 문서'를 위로 올림.
    """
    if not retrieved_docs:
        return []

    pairs = [[query, d.page_content] for d in retrieved_docs]
    scores = reranker.predict(pairs)

    order = np.argsort(scores)[::-1]
    out = []
    for idx in order[:top_k]:
        meta = dict(retrieved_docs[idx].metadata)
        meta["rerank_score"] = float(scores[idx])
        out.append(Document(page_content=retrieved_docs[idx].page_content, metadata=meta))
    return out

query = "월세 세액공제 한도가 얼마야? 2024년에 바뀐게 있어?"
cands = ensemble_retriever.invoke(query)
top3 = rerank_docs(query, cands, top_k=3)

for i, d in enumerate(top3, 1):
    print(f"Rank {i} | score={d.metadata.get('rerank_score'):.4f} | page={d.metadata.get('page')}")
    print(d.page_content[:140].replace("\n", " "), "...\n")


◻️ LLM(EXAONE) 로드 + RAG 체인(검색→재순위→생성)

In [ ]:
model_id = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",         # WHY: Colab 환경에서 GPU에 자동 배치
    trust_remote_code=True
)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.1,           # WHY: 세무 답변은 창의성보다 "정확/일관"이 중요
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=768,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

prompt = PromptTemplate.from_template(
"""
[역할]
너는 '2024년 귀속 연말정산 신고 안내' 문서를 근거로 답하는 세무 Q&A 도우미다.

[규칙]
- 아래 문서 조각(context)에 없는 내용은 추측하지 말 것
- 숫자/한도/요건은 표/문장 근거를 우선해서 답할 것
- 2024년 개정 내용이 있으면 그 내용을 우선해서 설명할 것
- 답은 짧은 문단 + 불릿으로 읽기 쉽게 작성할 것

[문서 근거]
{context}

[질문]
{question}

[답변]
"""
)

def answer(question: str, top_k: int = 3):
    # 1) 후보 검색
    retrieved = ensemble_retriever.invoke(question)

    # 2) 재순위
    final_docs = rerank_docs(question, retrieved, top_k=top_k)

    # 3) 컨텍스트 구성
    context = "\n\n".join([d.page_content for d in final_docs])

    # 4) 체인
    chain = (
        {"context": lambda _: context, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    result = chain.invoke(question)
    return result, final_docs

final_answer, sources = answer("월세 세액공제 한도가 얼마야? 2024년에 바뀐게 있어?", top_k=3)

print(final_answer)

print("\n--- 근거로 쓴 문서 ---")
for i, d in enumerate(sources, 1):
    print(f"[{i}] page={d.metadata.get('page')} score={d.metadata.get('rerank_score'):.4f} fmt={d.metadata.get('conversion','original')}")
    print(d.page_content[:160].replace("\n"," "), "...\n")


🟨 분석 결과

## 분석 결과 요약

모델 성능보다 먼저 문서 품질 + 검색 품질을 끌어올리는 것을 목표로 했다.  
세무 PDF는 표가 많고 반복 헤더가 많아서, 단순 RAG 구성으로는 정확도가 쉽게 무너진다.

---

### 1️⃣ 로더 선택 근거: PyMuPDF vs Upstage(HTML)

관찰한 문제:
- PyMuPDF로 로드하면 표가 텍스트로 평탄화되면서 열/행 의미가 사라질 수 있다.
- 큰 제목/목차 구조가 일부 약해져 “문서의 계층/맥락”이 희미해질 수 있다.

Upstage를 선택한 이유:
- Upstage는 HTML로 `<table>`, `<td>`, `rowspan/colspan`, `<h1>` 같은 구조를 보존한다.
- LLM은 구조 태그를 통해 “이게 표인지 / 각 열이 무엇을 의미하는지”를 더 정확히 파악할 수 있다.
→ 이후 모든 실험은 Upstage Document Parse(HTML) 기준으로 진행했다.

---

### 2️⃣ 문서 특성 분석: HTML 비중(태그)이 생각보다 큼

관찰한 문제:
- 페이지별로 HTML 길이와 순수 텍스트 길이를 비교했을 때, HTML 태그가 차지하는 비중이 크다.
- “태그는 많은데 텍스트는 거의 없는 페이지(정보 밀도 낮음)”가 존재한다.
- 이런 페이지는 토큰 낭비가 커서 LLM 비용/시간을 증가시키고, 청킹을 잘못하면 태그만 잘리는 문제도 생긴다.

대응 전략:
- 긴 청크(예: 2000자 이상) 중 HTML 태그가 많은 경우만 Markdown으로 변환(Smart Diet)한다.
- 짧은 청크까지 변환하지 않은 이유는, 변환 오차/레이아웃 깨짐 리스크가 더 커질 수 있기 때문이다.

---

### 3️⃣ 노이즈(헤더/푸터) 분석: 검색을 교란하는 반복 텍스트

관찰한 문제:
- 특정 헤더 문구가 수백 회 반복된다.
- 이런 반복 문구는 질문과 무관해도 BM25/임베딩 검색에서 상위로 뜰 가능성이 커진다.
- 즉, “문서 내용 검색”이 아니라 “헤더 검색”이 되는 상황이 생긴다.

대응 전략:
- 순수 텍스트 기준으로 50자 미만 페이지는 제거(쪽번호/서식/빈칸일 가능성이 큼)
- 반복 헤더 문구는 문자열 치환으로 제거(HTML 구조 자체는 최대한 유지)

---

### 4️⃣ 표/계층 구조 EDA: 세무 문서는 표 중심 문서

관찰:
- `<table>`이 다수 등장하며, 일부는 행이 매우 많은 “거대 표”도 존재한다.
- 거대 표는 고정 길이 기반 청킹(예: 512/1000)에서 중간이 찢어질 가능성이 높다.
- 헤더 태그는 문서에서 계층으로 완벽히 분리되지 않는 경우가 있어, “계층 기반 청킹”만으로 해결하기 어렵다.

결론:
- 표가 찢어지지 않도록 “의미 기반” 청킹을 우선 선택해야 한다.

---

### 5️⃣ 토큰 분포 분석: 단순 chunk_size는 위험

관찰:
- 페이지 토큰 수 중앙값이 높은 편이라, 512 기준으로 자르면 대부분 페이지가 2~3조각 이상으로 분해된다.
- 세무 문서는 한 페이지 안에서 정의/조건/예외/표가 이어지므로, 문맥 단절이 답변 정확도에 직접 타격을 준다.

대응:
- Semantic Chunking을 적용하고, 필요하면 “상한(max token)”을 두어 너무 큰 청크만 제한적으로 분할한다.

---

### 6️⃣ 임베딩 모델 선택 근거: BGE-m3

관찰:
- 주제별 샘플 문장을 임베딩 후 유사도 히트맵으로 비교했을 때,
  BGE-m3가 주제 내 응집(클러스터링)이 조금 더 또렷한 경향을 보였다.
- 다만 노이즈 텍스트에도 점수를 약간 후하게 주는 경향이 있어, 노이즈 제거가 중요하다.

결론:
- 세무 Q&A에서는 “관련 문서를 덩어리로 묶어내는 능력”이 중요하므로 BGE-m3를 채택했다.

---

### 7️⃣ 검색 전략: Hybrid(BM25 + Dense)로 안정성 확보

선택 이유:
- 세무 도메인은 용어 매칭이 중요하다(월세, 주택임차차입금 등).
- Dense 검색만 쓰면 키워드 정확도가 아쉬울 수 있고,
  Sparse(BM25)만 쓰면 표현 변화/동의어를 놓칠 수 있다.

결론:
- BM25 + Qdrant(Dense)를 앙상블로 결합(예: 0.3 / 0.7)하여 후보군의 재현율을 올렸다.

---

### 8️⃣ Reranking 도입: Top-K의 정밀도(Precision) 상승

관찰:
- Hybrid 검색 결과 중에는 “관련은 있지만 정답은 아닌 문서”가 섞일 수 있다.
- Reranker는 질문-문서 쌍을 더 강하게 비교해서,
  “질문 의도(예: 2024년 개정 여부)”에 맞는 문서를 상위로 끌어올린다.

결론:
- 후보군을 넓게 가져오고(Recall) → Reranker로 최종 Top-3를 엄선(Precision)하는 구조가 가장 안정적이었다.

---

### 9️⃣ 생성(LLM) 설정: EXAONE + 낮은 temperature

선택 이유:
- 세무 답변은 창의성보다 “사실/수치/근거”가 핵심이라 temperature를 낮게 유지한다.
- 4bit 양자화로 Colab 환경에서 구동 가능성을 확보한다.

프롬프트 원칙:
- 근거 문서에 없는 내용은 추측하지 않는다.
- 표/숫자는 근거를 우선하여 답한다.
- 2024년 개정 내용이 있으면 우선하여 설명한다.

---

## 🟨 결론
이 프로젝트의 성능은 LLM 교체보다,
1) 표 구조 보존(Upstage HTML)
2) 노이즈 제거
3) 의미 기반 청킹
4) Hybrid + Reranker
    - 위 네 가지가 더 크게 좌우했다.

다음 개선 후보:
- 거대 표 전용 처리(상한 토큰 설정 + 표 요약/압축)
- 인덱스 영구 저장(Qdrant 디스크 모드) 및 로딩 속도 최적화
- 답변에 “근거 페이지 번호/요약”을 더 깔끔하게 붙이는 출력 포맷 개선
